In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.cifar10.load_data()

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels))

In [ ]:
train_ds, val_ds = tf.keras.utils.split_dataset(
    train_ds, left_size=.8, right_size=.2, seed=1
)

In [ ]:
train_ds = train_ds.cache().prefetch(buffer_size=tf.data.experimental.AUTOTUNE).shuffle(buffer_size=train_ds.cardinality(), reshuffle_each_iteration=True).batch(64)
val_ds = val_ds.prefetch(buffer_size=tf.data.experimental.AUTOTUNE).shuffle(buffer_size=val_ds.cardinality(), reshuffle_each_iteration=True).batch(64)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
model = keras.Sequential([
  layers.Rescaling(1./255, input_shape=(32, 32, 3)),
  layers.Conv2D(128, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.BatchNormalization(),
  layers.Dropout(.3),
  layers.Conv2D(128, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.BatchNormalization(),
  layers.Dropout(.3),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.BatchNormalization(),
  layers.Dropout(.3),
  layers.Flatten(),
  layers.Dense(32, activation='relu'),
  layers.Dense(10, activation='softmax')
])

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

In [ ]:
test_ds = tf.data.Dataset.from_tensor_slices((test_images, test_labels))

In [ ]:
test_ds = test_ds.cache().prefetch(buffer_size=tf.data.experimental.AUTOTUNE).shuffle(buffer_size=test_ds.cardinality(), reshuffle_each_iteration=True).batch(64)

In [ ]:
epochs=50
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

In [ ]:
model.evaluate(test_ds)